## 1- Data Extraction

First thing to perfom is to extract the text from the provided PDFs and link each text with its physical page number. We assume that those files live inside the "*data/pdfs/Corpus_raw*" folder.

In [1]:
import ast
import pdfplumber
from typing import List, Dict, Tuple
import os
from tqdm import tqdm
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

In [2]:
def extract_pages_with_text(pdf_path: str) -> List[Tuple[int, str]]:
    """
    Returns list of (page_number, page_text)
    """
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if text and text.strip():
                pages.append((i, text))
    return pages

In [3]:
pdf_folder = "../data/pdfs/Corpus_raw"

In [ ]:
pdf_dicts = []
pdfs = os.listdir(pdf_folder)
for pdf_file in tqdm(pdfs, total=len(pdfs)):
    if pdf_file.endswith(".pdf"):
        pdf_path = os.path.join(pdf_folder, pdf_file)
        try:
            pages = extract_pages_with_text(pdf_path)
            pdf_dicts.append({"title": pdf_file, "text": pages})
        except Exception as e:
            print(pdf_path)
            print(str(e))
            pdf_dicts.append({"title": pdf_file, "text": ""})

The extracted texts can be saved in a *csv* file for further processing.

In [10]:
df_pdfs = pd.DataFrame.from_dict(pdf_dicts)
df_pdfs.to_csv(pdf_folder + "/extracted_pdfs.csv", sep="\t", encoding="utf-8", index=False)

## 2- Data Chunking and Indexation

Two chunking strategies were used in order to index the extracted documents: chunking by page and by paragraph

In [ ]:
extracted_pdfs = pd.read_csv(pdf_folder + "/extracted_pdfs.csv", sep="\t", encoding="utf-8")
extracted_pdfs.dropna(inplace=True)
extracted_pdfs.head()

In [ ]:
embedder = SentenceTransformer("dangvantuan/sentence-camembert-base")

chunk_size = 500
chunk_overlap = 50

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )

for chunking_strategy in ["by_page", "by_paragraph"]:
    # create Index
    chroma_client = chromadb.PersistentClient(path=f"../data/chroma_db_rag_{chunking_strategy}")
    collection = chroma_client.get_or_create_collection(name=f"rag_docs_{chunking_strategy}")
    
    for _, row in tqdm(extracted_pdfs.iterrows(), total=extracted_pdfs.shape[0]):
        texts = []
        metadatas = []
        ids = []

        title = row.title
        for page, text in ast.literal_eval(row.text):
            if chunking_strategy == "by_page":
                texts.append(text)
                metadatas.append(
                    {
                        "title": title,
                        "page_num": page
                    }
                )
                ids.append(f"{title}_page{page}")
            else:
                chunks = text_splitter.split_text(text)
                for chunk_id, chunk in enumerate(chunks):
                    texts.append(chunk)
                    metadatas.append(
                        {
                            "title": title,
                            "page_num": page
                        }
                    )
                    ids.append(f"{title}_page{page}_chunk{chunk_id}")
                    
        if texts:
            embeddings = embedder.encode(texts).tolist()
            
            collection.add(
                embeddings=embeddings,
                documents=texts,
                metadatas=metadatas,
                ids=ids
            )

## 3- Perform RAG

In [7]:
def perform_rag(query: str, top_k: int = 3, collection = collection) -> Dict:
    """
    """
    query_embedding = embedder.encode([query]).tolist()[0]
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas"]
    )
    
    retrieved_chunks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved_chunks.append({
            "text": doc,
            "page": meta["page_num"],
            "source":  meta["title"],
        })
    
    context_str = "\n\n---\n\n".join(
        [f"[Source: {chunk['source']}, Page {chunk['page']}]\n{chunk['text']}" for chunk in retrieved_chunks]
    )
    prompt = f"""You are a helpful assistant that answers questions strictly based on the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context_str}

Question: {query}

Answer in French (and mention the relevant source pages):"""

    response = ollama.chat(model="mistral", messages=[{"role": "user", "content": prompt}])
    answer = response["message"]["content"]
    
    return {
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }